In [1]:
# [Asistido por IA - Gemini]
import pandas as pd
import numpy as np

# Rutas relativas a los archivos que están en la raíz de tu repositorio
ruta_titanic = "Titanic-Dataset.csv"
ruta_telco = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

try:
    # Cargar los datasets
    df_titanic = pd.read_csv(ruta_titanic)
    df_telco = pd.read_csv(ruta_telco)
    
    print("¡Éxito! Datasets cargados correctamente.")
    print(f"Dimensiones Titanic: {df_titanic.shape}")
    print(f"Dimensiones Telco: {df_telco.shape}")
except FileNotFoundError as e:
    print(f"Error al cargar el archivo: {e}")
    print("Verifica que el nombre del archivo coincida exactamente con el explorador.")

¡Éxito! Datasets cargados correctamente.
Dimensiones Titanic: (891, 12)
Dimensiones Telco: (7043, 21)


In [2]:
from sklearn.model_selection import train_test_split

# --- Titanic: objetivo "Survived" ---
X_titanic = df_titanic.drop(columns=["Survived"])
y_titanic = df_titanic["Survived"]

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_titanic, y_titanic,
    test_size=0.2, random_state=42, stratify=y_titanic
)

print("TITANIC -> Train:", X_train_t.shape, "| Test:", X_test_t.shape)
print("% sobrevivientes en train:", round(y_train_t.mean() * 100, 1))
print("% sobrevivientes en test: ", round(y_test_t.mean() * 100, 1))

# --- Telco: objetivo "Churn" ---
X_telco = df_telco.drop(columns=["Churn"])
y_telco = df_telco["Churn"]

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_telco, y_telco,
    test_size=0.2, random_state=42, stratify=y_telco
)

print("\nTELCO -> Train:", X_train_c.shape, "| Test:", X_test_c.shape)
print(y_train_c.value_counts(normalize=True).round(3))
print(y_test_c.value_counts(normalize=True).round(3))

TITANIC -> Train: (712, 11) | Test: (179, 11)
% sobrevivientes en train: 38.3
% sobrevivientes en test:  38.5

TELCO -> Train: (5634, 20) | Test: (1409, 20)
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


In [3]:
import pandas as pd
import numpy as np

# Telco: TotalCharges viene como texto; los espacios vacíos pasan a NaN
for d in (X_train_c, X_test_c):
    d["TotalCharges"] = pd.to_numeric(d["TotalCharges"], errors="coerce")

print("NaN en TotalCharges (train):", X_train_c["TotalCharges"].isna().sum())

def resumen_outliers(serie, nombre, p_bajo=1, p_alto=99, factor=1.5):
    s = serie.dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    iqr_inf, iqr_sup = q1 - factor * iqr, q3 + factor * iqr
    per_inf, per_sup = s.quantile(p_bajo / 100), s.quantile(p_alto / 100)
    n_iqr = ((s < iqr_inf) | (s > iqr_sup)).sum()
    n_per = ((s < per_inf) | (s > per_sup)).sum()
    return {
        "variable": nombre,
        "min": s.min(), "max": s.max(),
        "IQR_inf": round(iqr_inf, 2), "IQR_sup": round(iqr_sup, 2),
        "outliers_IQR": n_iqr, "%_IQR": round(100 * n_iqr / len(s), 2),
        "P1": round(per_inf, 2), "P99": round(per_sup, 2),
        "outliers_perc": n_per, "%_perc": round(100 * n_per / len(s), 2),
    }

tabla = pd.DataFrame([
    resumen_outliers(X_train_t["Age"], "Titanic - Age"),
    resumen_outliers(X_train_t["Fare"], "Titanic - Fare"),
    resumen_outliers(X_train_c["MonthlyCharges"], "Telco - MonthlyCharges"),
    resumen_outliers(X_train_c["TotalCharges"], "Telco - TotalCharges"),
])
tabla

NaN en TotalCharges (train): 8


,variable,min,max,IQR_inf,IQR_sup,outliers_IQR,%_IQR,P1,P99,outliers_perc,%_perc
0,Titanic - Age,0.42,80.0000,-6.00,66.00,6,1.04,0.98,67.04,12,2.09
1,Titanic - Fare,0.00,512.3292,-26.76,65.66,91,12.78,0.00,227.52,7,0.98
2,Telco - MonthlyCharges,18.40,118.7500,-45.84,171.51,0,0.00,19.20,114.98,107,1.90
3,Telco - TotalCharges,18.85,8684.8000,-4739.73,8985.62,0,0.00,19.90,8040.22,112,1.99


In [4]:
print(tabla.to_string(index=False))

              variable   min       max  IQR_inf  IQR_sup  outliers_IQR  %_IQR    P1     P99  outliers_perc  %_perc
         Titanic - Age  0.42   80.0000    -6.00    66.00             6   1.04  0.98   67.04             12    2.09
        Titanic - Fare  0.00  512.3292   -26.76    65.66            91  12.78  0.00  227.52              7    0.98
Telco - MonthlyCharges 18.40  118.7500   -45.84   171.51             0   0.00 19.20  114.98            107    1.90
  Telco - TotalCharges 18.85 8684.8000 -4739.73  8985.62             0   0.00 19.90 8040.22            112    1.99


In [5]:
# Verificación: ¿qué son los pasajeros con Fare == 0? (solo train)
ceros = X_train_t[X_train_t["Fare"] == 0]

print("Cantidad de Fare == 0 en train:", len(ceros))
print("\nPor clase (Pclass):")
print(ceros["Pclass"].value_counts())
print("\nPor puerto (Embarked):")
print(ceros["Embarked"].value_counts(dropna=False))
print("\nPor sexo:")
print(ceros["Sex"].value_counts())
print("\nAge nulos entre ellos:", ceros["Age"].isna().sum())
print()
print(ceros[["Name", "Sex", "Age", "Pclass", "Embarked"]].to_string())

Cantidad de Fare == 0 en train: 14

Por clase (Pclass):
Pclass
2    6
1    5
3    3
Name: count, dtype: int64

Por puerto (Embarked):
Embarked
S    14
Name: count, dtype: int64

Por sexo:
Sex
male    14
Name: count, dtype: int64

Age nulos entre ellos: 8

                                 Name   Sex   Age  Pclass Embarked
481  Frost, Mr. Anthony Wood "Archie"  male   NaN       2        S
302   Johnson, Mr. William Cahoone Jr  male  19.0       3        S
732              Knight, Mr. Robert J  male   NaN       2        S
179               Leonard, Mr. Lionel  male  36.0       3        S
633     Parr, Mr. William Henry Marsh  male   NaN       1        S
466             Campbell, Mr. William  male   NaN       2        S
674        Watson, Mr. Ennis Hastings  male   NaN       2        S
597               Johnson, Mr. Alfred  male  49.0       3        S
815                  Fry, Mr. Richard  male   NaN       1        S
413    Cunningham, Mr. Alfred Fleming  male   NaN       2        S
263    

In [ ]:
print("Supervivencia con Fare == 0:", round(y_train_t.loc[ceros.index].mean() * 100, 1), "%")
print("Supervivencia general en train:", round(y_train_t.mean() * 100, 1), "%")


# Conclusión: los 14 casos con Fare == 0 son todos hombres de Southampton en las 3 clases,
# 8 con Age nulo. Parecen un grupo real (tarifa no pagada), no errores de captura.
# Decisión: no se tratan como outliers ni como faltantes. Winsorización solo en cola superior.

Supervivencia con Fare == 0: 0.0 %
Supervivencia general en train: 38.3 %


### 1. Clase Winsorizador Compatible con scikit-learn

In [11]:
# [Asistido por IA - Gemini, revisado]
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

class Winsorizador(BaseEstimator, TransformerMixin):
    def __init__(self, columnas=None, metodo="iqr", factor=1.5,
                 percentil=99, colas="superior"):
        self.columnas = columnas
        self.metodo = metodo
        self.factor = factor
        self.percentil = percentil
        self.colas = colas

    def fit(self, X, y=None):
        if self.metodo not in ("iqr", "percentil"):
            raise ValueError("metodo debe ser 'iqr' o 'percentil'")
        if self.colas not in ("superior", "ambas"):
            raise ValueError("colas debe ser 'superior' o 'ambas'")

        X_df = pd.DataFrame(X)
        cols = (list(self.columnas) if self.columnas is not None
                else X_df.select_dtypes(include=[np.number]).columns.tolist())
        faltantes = [c for c in cols if c not in X_df.columns]
        if faltantes:
            raise ValueError(f"Columnas no encontradas: {faltantes}")

        self.cols_a_procesar_ = cols
        self.columnas_entrada_ = np.asarray(X_df.columns, dtype=object)
        self.limites_ = {}   # se reinicia en cada fit

        for col in cols:
            serie = X_df[col].to_numpy(dtype=float)   # los NaN se ignoran abajo
            if self.metodo == "iqr":
                q1, q3 = np.nanquantile(serie, [0.25, 0.75])
                iqr = q3 - q1
                lim_inf, lim_sup = q1 - self.factor * iqr, q3 + self.factor * iqr
            else:
                lim_inf = np.nanquantile(serie, (100 - self.percentil) / 100)
                lim_sup = np.nanquantile(serie, self.percentil / 100)
            self.limites_[col] = {
                "inf": lim_inf if self.colas == "ambas" else -np.inf,
                "sup": lim_sup,
            }
        return self

    def transform(self, X):
        check_is_fitted(self, "limites_")
        X_df = pd.DataFrame(X).copy()
        faltantes = [c for c in self.cols_a_procesar_ if c not in X_df.columns]
        if faltantes:
            raise ValueError(f"Columnas no encontradas al transformar: {faltantes}")
        for col, lim in self.limites_.items():
            X_df[col] = X_df[col].clip(lower=lim["inf"], upper=lim["sup"])  # NaN intactos
        return X_df

    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        return self.columnas_entrada_

2. Título: Aplicación a Titanic y Telco (Límites Aprendidos)
Código:

In [12]:
# [Asistido por IA - Gemini]
# Instanciar transformadores
wins_titanic_iqr = Winsorizador(columnas=['Fare', 'Age'], metodo='iqr', factor=1.5, colas='superior')
wins_titanic_p99 = Winsorizador(columnas=['Fare', 'Age'], metodo='percentil', percentil=99, colas='superior')

wins_telco_iqr = Winsorizador(columnas=['MonthlyCharges'], metodo='iqr', factor=1.5, colas='superior')
wins_telco_p99 = Winsorizador(columnas=['MonthlyCharges'], metodo='percentil', percentil=99, colas='superior')

# Ajustar y transformar Train
X_train_t_iqr = wins_titanic_iqr.fit_transform(X_train_t)
X_train_t_p99 = wins_titanic_p99.fit_transform(X_train_t)

X_train_c_iqr = wins_telco_iqr.fit_transform(X_train_c)
X_train_c_p99 = wins_telco_p99.fit_transform(X_train_c)

# Transformar Test (sin ajustar)
X_test_t_iqr = wins_titanic_iqr.transform(X_test_t)
X_test_t_p99 = wins_titanic_p99.transform(X_test_t)
X_test_c_iqr = wins_telco_iqr.transform(X_test_c)
X_test_c_p99 = wins_telco_p99.transform(X_test_c)

def resumen_limites(nombre, wins, X_tr, X_te):
    print(f"\n{nombre}")
    for col, lim in wins.limites_.items():
        n_tr = int((X_tr[col] > lim["sup"]).sum())
        n_te = int((X_te[col] > lim["sup"]).sum())
        print(f"  {col}: límite sup = {lim['sup']:.2f} | recortados train = {n_tr} | test = {n_te}")

resumen_limites("Titanic IQR 1.5", wins_titanic_iqr, X_train_t, X_test_t)
resumen_limites("Titanic P99",     wins_titanic_p99, X_train_t, X_test_t)
resumen_limites("Telco IQR 1.5",   wins_telco_iqr,   X_train_c, X_test_c)
resumen_limites("Telco P99",       wins_telco_p99,   X_train_c, X_test_c)


Titanic IQR 1.5
  Fare: límite sup = 65.66 | recortados train = 91 | test = 25
  Age: límite sup = 66.00 | recortados train = 6 | test = 1

Titanic P99
  Fare: límite sup = 227.53 | recortados train = 7 | test = 4
  Age: límite sup = 67.04 | recortados train = 6 | test = 1

Telco IQR 1.5
  MonthlyCharges: límite sup = 171.51 | recortados train = 0 | test = 0

Telco P99
  MonthlyCharges: límite sup = 114.98 | recortados train = 57 | test = 8


3. Título: Verificación Anti-Fuga de Información
Código:

In [13]:
# [Asistido por IA - Gemini, revisado]
def lim_iqr(s, f=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    return q3 + f * (q3 - q1)

lim_clase = wins_titanic_iqr.limites_["Fare"]["sup"]          # lo que realmente se aplicó
lim_train = lim_iqr(X_train_t["Fare"])                        # recalculado solo con train
lim_todo  = lim_iqr(pd.concat([X_train_t, X_test_t])["Fare"]) # SOLO ilustrativo: así se vería la fuga

print(f"Límite aprendido por la clase : {lim_clase:.4f}")
print(f"Recalculado solo con train    : {lim_train:.4f}")
print(f"Con todo el dataset (ilustra) : {lim_todo:.4f}  (diferencia: {abs(lim_todo - lim_clase):.4f})")

# 1) La clase usó solo train
assert np.isclose(lim_clase, lim_train), "La clase NO usó solo train"

# 2) El test se recortó con el límite de TRAIN (no con uno propio)
esperado = min(X_test_t["Fare"].max(), lim_clase)
assert np.isclose(X_test_t_iqr["Fare"].max(), esperado), "El test no usó el límite de train"

# 3) Integridad: filas, índice, NaN y ceros intactos, original sin modificar
for orig, trans in [(X_train_t, X_train_t_iqr), (X_test_t, X_test_t_iqr)]:
    assert orig.shape == trans.shape and orig.index.equals(trans.index)
    assert orig["Age"].isna().sum() == trans["Age"].isna().sum()
    assert (orig["Fare"] == 0).sum() == (trans["Fare"] == 0).sum()
assert X_train_t["Fare"].max() > lim_clase     # el DataFrame original no se modificó
print("Verificaciones anti-fuga e integridad: OK")

Límite aprendido por la clase : 65.6563
Recalculado solo con train    : 65.6563
Con todo el dataset (ilustra) : 65.6344  (diferencia: 0.0219)
Verificaciones anti-fuga e integridad: OK


4. Título: Comparación Estadística Antes/Después
Código:

In [14]:
# [Asistido por IA - Gemini, revisado]
def generar_comparacion(original, escenarios, limites, columna):
    """original: DataFrame sin tratar (train). escenarios: {nombre: DataFrame winsorizado}.
    limites: {nombre: límite superior aprendido por la clase}."""
    def resumen(s):
        s = s.dropna()
        return {"n": len(s), "Media": s.mean(), "Mediana": s.median(),
                "Desv_Est": s.std(), "Mínimo": s.min(), "Máximo": s.max(),
                "Asimetría": s.skew()}

    filas = {"Original": {**resumen(original[columna]),
                          "Límite_sup": np.nan, "Recortados": 0}}
    for nombre, df in escenarios.items():
        lim = limites[nombre]
        filas[nombre] = {**resumen(df[columna]), "Límite_sup": lim,
                         "Recortados": int((original[columna] > lim).sum())}

    tabla = pd.DataFrame(filas).T
    for c in ["Media", "Desv_Est"]:
        tabla[f"Δ%_{c}"] = 100 * (tabla[c] / tabla.loc["Original", c] - 1)
    return tabla.round(2)

# Titanic
esc_t = {"IQR 1.5": X_train_t_iqr, "P99": X_train_t_p99}
for col in ["Fare", "Age"]:
    lims = {"IQR 1.5": wins_titanic_iqr.limites_[col]["sup"],
            "P99": wins_titanic_p99.limites_[col]["sup"]}
    print(f"\n--- Titanic ({col}) ---")
    print(generar_comparacion(X_train_t, esc_t, lims, col).to_string())

# Telco
esc_c = {"IQR 1.5": X_train_c_iqr, "P99": X_train_c_p99}
lims_c = {"IQR 1.5": wins_telco_iqr.limites_["MonthlyCharges"]["sup"],
          "P99": wins_telco_p99.limites_["MonthlyCharges"]["sup"]}
print("\n--- Telco (MonthlyCharges) ---")
print(generar_comparacion(X_train_c, esc_c, lims_c, "MonthlyCharges").to_string())


--- Titanic (Fare) ---
              n  Media  Mediana  Desv_Est  Mínimo  Máximo  Asimetría  Límite_sup  Recortados  Δ%_Media  Δ%_Desv_Est
Original  712.0  31.82    14.45     48.06     0.0  512.33       4.65         NaN         0.0      0.00         0.00
IQR 1.5   712.0  23.96    14.45     20.51     0.0   65.66       1.08       65.66        91.0    -24.71       -57.32
P99       712.0  30.82    14.45     41.00     0.0  227.52       2.93      227.52         7.0     -3.15       -14.70

--- Titanic (Age) ---
              n  Media  Mediana  Desv_Est  Mínimo  Máximo  Asimetría  Límite_sup  Recortados  Δ%_Media  Δ%_Desv_Est
Original  575.0  29.81     28.5     14.49    0.42   80.00       0.35         NaN         0.0      0.00         0.00
IQR 1.5   575.0  29.74     28.5     14.29    0.42   66.00       0.26       66.00         6.0     -0.24        -1.37
P99       575.0  29.75     28.5     14.31    0.42   67.04       0.27       67.04         6.0     -0.20        -1.18

--- Telco (MonthlyCharge